# From Pandas to PySpark: Superstore ETL

**Part 1** uses the pandas operations you already know (`head`, `info`, `value_counts`, filter, `groupby`) and shows the Spark version of each.

**Part 2** builds an end-to-end ETL: the dirty Superstore CSV is cleaned and split into a **star schema** (dimension & fact tables), then loaded into PostgreSQL.

Data: `/workspace/data/superstore_raw.csv` (1,436 rows of Superstore orders, with some dirty data on purpose).

### What is Spark?
- pandas works on **one computer** and keeps all data in memory.
- Spark splits the data into **partitions** and processes them **in parallel**, so it can handle data that is too big for pandas.
- The code looks very similar to pandas. The two biggest differences:
  1. Spark is **lazy**: it only computes when you ask for a result (`show`, `count`, `write`).
  2. A Spark DataFrame **cannot be changed in place**. Always save the result: `df = df.withColumn(...)`.

### SETUP

In [ ]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("superstore-etl").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")   # hide Spark warning messages
spark

- The first run prints a few `WARNING` lines while Spark starts. That is normal.
- `SparkSession` is the entry point to Spark, like `import pandas as pd`.
- `functions as F` holds Spark's column functions (`F.col`, `F.sum`, `F.trim`, ...).
- While the session is running, open the Spark UI at http://localhost:4040.

# Part 1: Pandas to Spark

## Create DataFrame

pandas: `pd.DataFrame(data)` → Spark: `spark.createDataFrame(...)`. A pandas DataFrame can be converted directly.

In [ ]:
data = {'Name': ['Alice', 'Bob', 'Charlie'],
        'Age': [25, 30, 22],
        'City': ['New York', 'London', 'Paris']}

pdf = pd.DataFrame(data)          # pandas
sdf = spark.createDataFrame(pdf)  # Spark
sdf.show()

## Loading Data

pandas: `pd.read_csv('file.csv')` → Spark: `spark.read.csv('file.csv', header=True, inferSchema=True)`

- `header=True`: the first row is the column names (pandas does this by default).
- `inferSchema=True`: let Spark detect the data types (pandas does this by default).
- `escape='"'`: some product names contain commas and quotes, for example `Chair, "Executive Red"`.
  pandas handles this automatically; Spark needs this option, otherwise the columns shift.
- The path starts with `/workspace/...` because the notebook runs inside the Docker container.

In [ ]:
raw = spark.read.csv('/workspace/data/superstore_raw.csv', header=True, inferSchema=True, escape='"')

## Simple Exploring Data

**Show first 5 rows** → pandas: `df.head()`

In [ ]:
raw.show(5)

Too wide? Convert a few rows to pandas to get the familiar table:

In [ ]:
raw.limit(5).toPandas()

**Number of rows and columns** → pandas: `df.shape`

In [ ]:
(raw.count(), len(raw.columns))

**DataFrame summary** → pandas: `df.info()`

In [ ]:
raw.printSchema()

**Descriptive statistics** → pandas: `df.describe()`

In [ ]:
raw.describe('Sales', 'Quantity', 'Discount', 'Profit').show()

`Sales` has fewer values than the other columns, so some values are missing.

**Column list and data types** → pandas: `df.columns`, `df.dtypes`

In [ ]:
raw.columns

In [ ]:
raw.dtypes

**Number of unique values** → pandas: `df.nunique()`

In [ ]:
raw.select(F.countDistinct('Order ID'), F.countDistinct('Customer ID'), F.countDistinct('Product ID')).show()

**Frequency count** → pandas: `df['Segment'].value_counts()`

In [ ]:
raw.groupBy('Segment').count().orderBy(F.desc('count')).show()

`consumer ` and `Consumer` are the same segment written differently. This is dirty data we will clean in Part 2.

**Unique values** → pandas: `df['Region'].unique()`

In [ ]:
raw.select('Region').distinct().show()

## Basic Data Manipulation

**Sorting** → pandas: `df.sort_values(by='Sales', ascending=False)`

In [ ]:
raw.orderBy(F.desc('Sales')).select('Order ID', 'Product Name', 'Sales').show(5, truncate=False)

**Accessing columns** → pandas: `df[['Order ID', 'Sales']]`

In [ ]:
raw.select('Order ID', 'Sales').show(3)

**Filtering, single condition** → pandas: `df[df['Sales'] > 3000]`

In [ ]:
raw.filter(F.col('Sales') > 3000).select('Order ID', 'Category', 'Sales').show()

**Filtering, multiple conditions** → pandas: `df[(df['Sales'] > 1000) & (df['Region'] == 'West')]`

In [ ]:
raw.filter((F.col('Sales') > 1000) & (F.col('Region') == 'West')).select('Order ID', 'Region', 'Sales').show(5)

**Query string** → pandas: `df.query('Sales > 3000')`

In [ ]:
raw.filter('Sales > 3000').count()

**isin** → pandas: `df[df['State'].isin(['California', 'Texas'])]`

In [ ]:
raw.filter(F.col('State').isin('California', 'Texas')).groupBy('State').count().show()

**Text contains** → pandas: `df[df['Product Name'].str.contains('Chair')]`

In [ ]:
raw.filter(F.col('Product Name').contains('Chair')).select('Product Name').show(3, truncate=False)

**Add a new column** → pandas: `df['Unit Price'] = df['Sales'] / df['Quantity']`

Spark cannot change a DataFrame in place, so the result must be saved to a variable.

In [ ]:
with_price = raw.withColumn('Unit Price', F.round(F.col('Sales') / F.col('Quantity'), 2))
with_price.select('Sales', 'Quantity', 'Unit Price').show(3)

**Remove a column** → pandas: `df.drop(columns='Unit Price')`

In [ ]:
with_price.drop('Unit Price').columns[-4:]

**Grouping and aggregation** → pandas: `df.groupby('Category').agg({'Sales': 'sum', 'Order ID': 'count'})`

In [ ]:
(raw.groupBy('Category')
    .agg(F.round(F.sum('Sales'), 2).alias('Total Sales'),
         F.count('Order ID').alias('Rows'))
    .orderBy(F.desc('Total Sales'))
    .show())

Notice ` furniture` next to `Furniture`: the same dirty-text problem again.

## Pandas vs Spark summary

| pandas | Spark |
| --- | --- |
| `pd.read_csv(path)` | `spark.read.csv(path, header=True, inferSchema=True, escape='"')` |
| `df.head()` | `df.show(5)` |
| `df.shape` | `(df.count(), len(df.columns))` |
| `df.info()` | `df.printSchema()` |
| `df['col'].value_counts()` | `df.groupBy('col').count()` |
| `df['col'].unique()` | `df.select('col').distinct()` |
| `df.sort_values('col')` | `df.orderBy('col')` |
| `df[['a', 'b']]` | `df.select('a', 'b')` |
| `df[df['a'] > 1]` | `df.filter(F.col('a') > 1)` |
| `df['new'] = ...` | `df = df.withColumn('new', ...)` |
| `df.drop(columns='a')` | `df.drop('a')` |
| `df.groupby('a').agg(...)` | `df.groupBy('a').agg(...)` |
| (already pandas) | `df.toPandas()` |

# Part 2: ETL Superstore → Star Schema

The raw file is one big **flat table**: every row repeats the customer name, product name, city, etc.
We will split it into a **star schema**:

```
                 dim_customer
                      │
   dim_product ── fact_sales ── dim_location
                      │
                   dim_date
```

- **Dimension table** (`dim_`): describes *who / what / where / when*. One row per customer, product, location, date.
- **Fact table** (`fact_`): the events we measure (sales, quantity, profit), with IDs pointing to the dimensions.

Steps: **E**xtract (read CSV) → **T**ransform (clean, build tables) → **L**oad (write to PostgreSQL).

## E: Extract

In [ ]:
raw = spark.read.csv('/workspace/data/superstore_raw.csv', header=True, inferSchema=True, escape='"')
print('rows:', raw.count())

## T: Transform

### Step 1. Rename columns to snake_case

`Customer ID` → `customer_id`. Column names without spaces are easier to use in Spark and in SQL.
pandas: `df.columns = [...]` → Spark: `df.toDF(*new_names)`

In [ ]:
new_names = [c.lower().replace(' ', '_').replace('-', '_').replace('/', '_') for c in raw.columns]
df = raw.toDF(*new_names)
df.columns

### Step 2. Check data quality

**Duplicate rows** → pandas: `df.duplicated().sum()`

In [ ]:
df.count() - df.dropDuplicates().count()

**Missing values per column** → pandas: `df.isna().sum()`

In [ ]:
for c in df.columns:
    n = df.filter(F.col(c).isNull()).count()
    if n > 0:
        print(c, n)

**Inconsistent text**

In [ ]:
df.groupBy('segment').count().show()
df.groupBy('category').count().show()

**Postal codes that lost their leading zero** (`6824` should be `06824`)

In [ ]:
df.filter(F.length(F.col('postal_code').cast('string')) < 5).select('city', 'state', 'postal_code').show(3)

**Dates are still text**, in `month/day/year` format:

In [ ]:
df.select('order_date', 'ship_date').printSchema()

### Step 3. Clean

| Problem | Fix |
| --- | --- |
| duplicate rows | `dropDuplicates()` |
| extra spaces & mixed case | `F.trim()` + `F.initcap()` (like `.str.strip().str.title()`) |
| postal code without leading zero | cast to text + `F.lpad(..., 5, '0')` |
| missing postal code | fill with `'Unknown'` |
| dates as text | `F.to_date(col, 'M/d/yyyy')` (like `pd.to_datetime`) |
| missing sales | drop the row, it cannot be used as a fact |

In [ ]:
clean = (
    df.dropDuplicates()
      .withColumn('customer_name', F.trim('customer_name'))
      .withColumn('segment', F.initcap(F.trim('segment')))
      .withColumn('region', F.initcap(F.trim('region')))
      .withColumn('category', F.initcap(F.trim('category')))
      .withColumn('postal_code', F.lpad(F.col('postal_code').cast('string'), 5, '0'))
      .fillna({'postal_code': 'Unknown'})
      .withColumn('order_date', F.to_date('order_date', 'M/d/yyyy'))
      .withColumn('ship_date', F.to_date('ship_date', 'M/d/yyyy'))
      .dropna(subset=['sales'])
)

print('before:', df.count(), '| after:', clean.count())
clean.groupBy('segment').count().show()

### Step 4. Build the dimension tables

A dimension keeps **one row per ID**: `select` the columns, then `dropDuplicates` on the ID.

In [ ]:
dim_customer = clean.select('customer_id', 'customer_name', 'segment').dropDuplicates(['customer_id'])
dim_customer.show(3)

Some product IDs have more than one product name in the raw data:

In [ ]:
clean.groupBy('product_id').agg(F.countDistinct('product_name').alias('names')).filter('names > 1').show(3)

`dropDuplicates(['product_id'])` keeps one name for each ID, so every product ID appears only once.

In [ ]:
dim_product = (clean.select('product_id', 'product_name', 'category', 'sub_category')
                    .dropDuplicates(['product_id']))
dim_product.show(3, truncate=False)

A location has no ID in the raw data, so we create one: take the unique locations, sort them, and number them 1, 2, 3, ...

In [ ]:
location_cols = ['country_region', 'region', 'state', 'city', 'postal_code']

dim_location = (clean.select(location_cols).dropDuplicates()
                     .withColumn('location_id', F.row_number().over(Window.orderBy('state', 'city', 'postal_code'))))
dim_location.show(3)

The date dimension has one row per date, with parts that are useful for reports.
`date_id` is the date as a number, for example `20201108`.

In [ ]:
dates = clean.select(F.col('order_date').alias('date')).union(clean.select(F.col('ship_date').alias('date'))).distinct()

dim_date = (dates
    .withColumn('date_id', F.date_format('date', 'yyyyMMdd').cast('int'))
    .withColumn('year', F.year('date'))
    .withColumn('quarter', F.quarter('date'))
    .withColumn('month', F.month('date'))
    .withColumn('month_name', F.date_format('date', 'MMMM'))
    .withColumn('day_name', F.date_format('date', 'EEEE')))
dim_date.orderBy('date').show(3)

### Step 5. Build the fact table

The fact table keeps the numbers (`sales`, `quantity`, `discount`, `profit`) and only the **IDs** of each dimension.
`location_id` is taken by joining back to `dim_location`.

In [ ]:
fact_sales = (clean
    .join(dim_location, on=location_cols, how='left')
    .withColumn('order_date_id', F.date_format('order_date', 'yyyyMMdd').cast('int'))
    .withColumn('ship_date_id', F.date_format('ship_date', 'yyyyMMdd').cast('int'))
    .select('row_id', 'order_id', 'order_date_id', 'ship_date_id', 'customer_id', 'product_id',
            'location_id', 'ship_mode', 'sales', 'quantity', 'discount', 'profit'))
fact_sales.show(3)

**Check:** one fact row per clean row, and no missing IDs.

In [ ]:
print('clean rows:', clean.count(), '| fact rows:', fact_sales.count())
print('missing location_id:', fact_sales.filter(F.col('location_id').isNull()).count())

for name, table in [('dim_customer', dim_customer), ('dim_product', dim_product),
                    ('dim_location', dim_location), ('dim_date', dim_date), ('fact_sales', fact_sales)]:
    print(name, table.count())

## L: Load to PostgreSQL

The tables are written to **PostgreSQL on your laptop** (the one you open in pgAdmin).

| | Host | Port |
| --- | --- | --- |
| pgAdmin (on the laptop) | `localhost` | `5432` |
| This notebook (inside the container) | `host.docker.internal` | `5432` |

Inside the container, `localhost` means the container itself, so we use `host.docker.internal` to reach the laptop.

The host, user and password come from the `.env` file, so the password is **never written in the notebook**.

In [ ]:
import os

missing = [k for k in ('PG_HOST', 'PG_PORT', 'PG_DB', 'PG_USER', 'PG_PASSWORD') if not os.getenv(k)]
if missing:
    raise SystemExit(f'Not set in .env: {missing}. Copy .env.example to .env, fill it in, then run: docker compose up -d')

jdbc_url = f"jdbc:postgresql://{os.environ['PG_HOST']}:{os.environ['PG_PORT']}/{os.environ['PG_DB']}"
jdbc_props = {'user': os.environ['PG_USER'], 'password': os.environ['PG_PASSWORD'], 'driver': 'org.postgresql.Driver'}
print(jdbc_url)

**Test the connection** before writing:

In [ ]:
import psycopg2

try:
    conn = psycopg2.connect(host=os.environ['PG_HOST'], port=os.environ['PG_PORT'], dbname=os.environ['PG_DB'],
                            user=os.environ['PG_USER'], password=os.environ['PG_PASSWORD'], connect_timeout=5)
    conn.close()
    print('Connected to PostgreSQL')
except psycopg2.OperationalError as e:
    msg = str(e)
    if 'does not exist' in msg:
        raise SystemExit(f"Database {os.environ['PG_DB']} does not exist. Create it in pgAdmin first.")
    if 'password authentication failed' in msg:
        raise SystemExit('Wrong password. PG_USER / PG_PASSWORD in .env must match pgAdmin.')
    if 'Connection refused' in msg or 'timeout' in msg:
        raise SystemExit('Cannot reach PostgreSQL on the laptop. Make sure it is running (pgAdmin can connect).')
    raise

**Write the tables.** Dimensions first, then the fact table.

`mode='overwrite'` drops and recreates the table each time, which is convenient while practising.
For GC6, create the tables with a DDL script first (with primary & foreign keys), then load with `mode='append'`.

In [ ]:
tables = [('dim_customer', dim_customer), ('dim_product', dim_product), ('dim_location', dim_location),
          ('dim_date', dim_date), ('fact_sales', fact_sales)]

for name, table in tables:
    table.write.jdbc(jdbc_url, name, mode='overwrite', properties=jdbc_props)
    print('loaded', name)

## Check the result

Read back from PostgreSQL with a query that joins the fact table to two dimensions.
The same query can be run in pgAdmin (database → Query Tool).

In [ ]:
query = '''
(SELECT l.region, p.category, ROUND(SUM(f.sales)::numeric, 2)::float AS total_sales
 FROM fact_sales f
 JOIN dim_location l ON f.location_id = l.location_id
 JOIN dim_product  p ON f.product_id  = p.product_id
 GROUP BY l.region, p.category
 ORDER BY l.region, p.category) AS t
'''
spark.read.jdbc(jdbc_url, query, properties=jdbc_props).show()

Done. Stop Spark to free the laptop's memory.

In [ ]:
spark.stop()